In [ ]:

# %pip install -U openai httpx pandas tqdm pydantic python-dotenv

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
import os
import json
import time
import random
from typing import List, Literal

import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field

from openai import OpenAI


DATA_PATH = "/Users/IvyGuo/Documents/GitHub/AIL-sensory-fair/Sensory v1  - sensoryfair_v1.csv" 
OUT_JSONL = "llm_feature_tags.jsonl"
OUT_CSV = "llm_feature_tags.csv"

MODEL = "gpt-4o-mini"
TEMPERATURE = 0

# Rate-limit helpers (tune as needed)
BASE_DELAY_S = 3.0          # sleep between requests (set higher if still 429)
MAX_RETRIES = 6             # retries on transient errors (429/5xx)
BACKOFF_BASE = 1.8          # exponential backoff multiplier
JITTER_S = 0.25             # random jitter to avoid bursts

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set in your environment."
client = OpenAI()

print("Setup OK")

Setup OK


In [41]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="Say hello in one short sentence."
)

print(response.output[0].content[0].text)

Hello!


In [25]:
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

Rows: 52
Columns: ['product_id', 'url_hash', 'source', 'scraped_at', 'product_name', 'product_url', 'short_description', 'full_description', 'category', 'text_for_labeling', 'seamless', 'fabric_softness', 'wool_present', 'texture_level', 'pressure_level', 'adaptive_or_sensory_claim', 'seamless_evidence', 'fabric_softness_evidence', 'wool_evidence', 'texture_evidence', 'pressure_evidence', 'adaptive_claim_evidence', 'labeler', 'label_status', 'review_notes']


,product_id,url_hash,source,scraped_at,product_name,product_url,short_description,full_description,category,text_for_labeling,...,adaptive_or_sensory_claim,seamless_evidence,fabric_softness_evidence,wool_evidence,texture_evidence,pressure_evidence,adaptive_claim_evidence,labeler,label_status,review_notes
0,1,a5debd253afe,sensoryfair.com,NaN,Abby & Noah Sensory Compression Vest for Women...,https://sensoryfair.com/product/abby-noah-sens...,Therapist-Designed Compression Hug: Our “weara...,Therapist-Designed Compression Hug: Our “weara...,NaN,product_name: Abby & Noah Sensory Compression ...,...,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unlabeled,NaN
1,2,eae004a58e62,sensoryfair.com,NaN,Snugabye Adaptive Back Zip Convert-A-Foot Slee...,https://sensoryfair.com/product/adaptive-back-...,NaN,This collection was designed with your special...,NaN,product_name: Snugabye Adaptive Back Zip Conve...,...,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unlabeled,NaN
2,3,e2520a1845e7,sensoryfair.com,NaN,"CBObaby Adaptive Bodysuit, Short Sleeve, Front...",https://sensoryfair.com/product/adaptive-bodys...,NaN,Discover comfort and functionality with this i...,NaN,"product_name: CBObaby Adaptive Bodysuit, Short...",...,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unlabeled,NaN


In [27]:
def get_text_fields(row: pd.Series) -> dict:
    return {
        "product_name": row.get("product_name", ""),
        "short_description": row.get("short_description", ""),
        "full_description": row.get("full_description", ""),
        "product_url": row.get("product_url", ""),
        "price": row.get("price", ""),
    }

def coerce_product_id(row: pd.Series, idx: int) -> str:
    for k in ["product_id", "id", "sku", "handle"]:
        if k in row and pd.notna(row[k]):
            return str(row[k])
    return str(idx)


In [29]:
Tri = Literal["yes", "no", "unknown"]
Quad = Literal["low", "medium", "high", "unknown"]

class BinaryFeature(BaseModel):
    value: Tri
    evidence: List[str] = Field(default_factory=list)

class OrdinalFeature(BaseModel):
    value: Quad
    evidence: List[str] = Field(default_factory=list)

class FeatureTagRecord(BaseModel):
    product_id: str

    seamless: BinaryFeature
    fabric_softness: OrdinalFeature
    wool_present: BinaryFeature
    texture_level: OrdinalFeature
    pressure_level: OrdinalFeature
    adaptive_or_sensory_claim: BinaryFeature

    review_notes: str = ""
    confidence: float = Field(..., ge=0.0, le=1.0)


## Prompt

In [32]:
SYSTEM_INSTRUCTIONS = """You are a strict product feature tagger for sensory-relevant attributes.
Your task is ONLY to tag the requested features. Do not classify overall sensory-friendliness.

CRITICAL RULES:
1) NO INFERENCE: Only label based on explicit text in the provided descriptions. Do not use outside knowledge.
2) EVIDENCE REQUIRED: Every non-unknown label MUST include at least one verbatim quote from the description.
3) CONTRADICTIONS: If equally strong conflicting signals exist, label unknown and record both phrases in review_notes.
4) PRECEDENCE: technical terms > strong modifiers > fit/style descriptors > generic comfort words.
5) DO NOT GUESS: If missing, use unknown.
6) Output MUST be valid JSON matching the schema exactly. No extra keys."""

LABEL_GUIDE = """REFERENCE (for consistency; still no inference):
Tactile concerns often involve seams, fabric softness, texture, wool irritation, and compression/pressure.
Only tag what is explicitly stated."""

DECISION_HINTS = """FEATURE DECISION HINTS (use as a decision tree):
SEAMLESS (yes/no/unknown):
- yes if: seamless, no seams, no side seams, flatlock seams, flat seams, bonded seams, bonded construction (re seam removal)
- no if: reinforced seams, double stitched seams, exposed seams

FABRIC_SOFTNESS (low/medium/high/unknown):
- high if: ultra soft, super soft, buttery soft, extra soft, brushed interior, plush
- medium if: soft, comfortable, smooth fabric
- low if: stiff, structured fabric, firm material, heavy canvas, durable canvas
(If equally strong conflicting phrases, choose unknown)

WOOL_PRESENT (yes/no/unknown):
- yes if: wool, merino wool, wool blend
- no if: a full material composition is listed and wool is not included (e.g., 100% cotton)

TEXTURE_LEVEL (low/medium/high/unknown):
- high if: ribbed, waffle knit, cable knit, textured, raised pattern, embossed, embroidered texture
- medium if: knit (visible knit structure), patterned, heathered
- low if: smooth, flat surface, no texture
(Technical texture terms override generic comfort words.)

PRESSURE_LEVEL (low/medium/high/unknown):
- high if: firm compression, strong compression, compression therapy, graduated compression
- medium if: compression, supportive, snug fit, body-hugging, slim fit, form-fitting
- low if: loose fit, relaxed fit, flowy, oversized

ADAPTIVE_OR_SENSORY_CLAIM (yes/no/unknown):
- yes if: sensory friendly, sensory-friendly, adaptive clothing, autism friendly, special needs clothing, designed for sensory comfort
- no if: explicitly not adaptive / not sensory-focused
"""

def build_user_prompt(product: dict) -> str:
    name = product.get("product_name","") or ""
    short_desc = product.get("short_description","") or ""
    full_desc = product.get("full_description","") or ""
    url = product.get("product_url","") or ""
    price = str(product.get("price","") or "")

    return f"""{LABEL_GUIDE}

{DECISION_HINTS}

PRODUCT:
name: {name}
url: {url}
price: {price}

short_description:
{short_desc}

full_description:
{full_desc}

Return JSON matching the schema exactly.
"""


In [34]:
def _sleep_with_jitter(base: float):
    time.sleep(base + random.random() * JITTER_S)

def tag_one(product: dict, product_id: str) -> FeatureTagRecord:
    user_prompt = build_user_prompt(product)

    last_err = None
    delay = BASE_DELAY_S

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.parse(
                model=MODEL,
                instructions=SYSTEM_INSTRUCTIONS,
                input=[{"role": "user", "content": [{"type": "input_text", "text": user_prompt}]}],
                text_format=FeatureTagRecord,
                temperature=TEMPERATURE,
            )
            rec: FeatureTagRecord = resp.output_parsed
            rec.product_id = product_id
            return rec

        except Exception as e:
            last_err = e
            msg = str(e)

            # Backoff on rate limits or transient server issues
            if "429" in msg or "Rate limit" in msg or "503" in msg or "500" in msg or "502" in msg:
                _sleep_with_jitter(delay)
                delay *= BACKOFF_BASE
                continue

            raise

    raise RuntimeError(f"Failed after {MAX_RETRIES} attempts: {last_err}")


## Run batch tagging

Start small (`N_LIMIT = 25`) to confirm everything works and to avoid 429s. Then scale up.


In [47]:
# -----------------------------
# Batch run
# -----------------------------
N_LIMIT = None  # <-- set to None for full dataset once stable

rows = df if N_LIMIT is None else df.head(N_LIMIT)

jsonl_lines = []
out_rows = []

for idx, row in tqdm(list(rows.iterrows()), total=len(rows)):
    prod = get_text_fields(row)
    pid = coerce_product_id(row, idx)

    try:
        rec = tag_one(prod, pid)
        d = rec.model_dump()
        jsonl_lines.append(json.dumps(d, ensure_ascii=False))

        out_rows.append({
            "product_id": pid,
            "product_name": prod.get("product_name", ""),
            "product_url": prod.get("product_url", ""),

            "seamless": d["seamless"]["value"],
            "fabric_softness": d["fabric_softness"]["value"],
            "wool_present": d["wool_present"]["value"],
            "texture_level": d["texture_level"]["value"],
            "pressure_level": d["pressure_level"]["value"],
            "adaptive_or_sensory_claim": d["adaptive_or_sensory_claim"]["value"],

            # evidence columns (optional but great for audit)
            "seamless_evidence": " | ".join(d["seamless"]["evidence"]),
            "fabric_softness_evidence": " | ".join(d["fabric_softness"]["evidence"]),
            "wool_present_evidence": " | ".join(d["wool_present"]["evidence"]),
            "texture_level_evidence": " | ".join(d["texture_level"]["evidence"]),
            "pressure_level_evidence": " | ".join(d["pressure_level"]["evidence"]),
            "adaptive_claim_evidence": " | ".join(d["adaptive_or_sensory_claim"]["evidence"]),

            "confidence": d["confidence"],
            "review_notes": d.get("review_notes", ""),
        })

        _sleep_with_jitter(BASE_DELAY_S)

    except Exception as e:
        err_obj = {
            "product_id": pid,
            "error": str(e),
            "product_name": prod.get("product_name", ""),
            "product_url": prod.get("product_url", ""),
        }
        jsonl_lines.append(json.dumps(err_obj, ensure_ascii=False))

        out_rows.append({
            "product_id": pid,
            "product_name": prod.get("product_name", ""),
            "product_url": prod.get("product_url", ""),
            "seamless": "",
            "fabric_softness": "",
            "wool_present": "",
            "texture_level": "",
            "pressure_level": "",
            "adaptive_or_sensory_claim": "",
            "confidence": None,
            "review_notes": "",
            "error": str(e),
        })

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    f.write("\n".join(jsonl_lines))

out_df = pd.DataFrame(out_rows)
out_df.to_csv(OUT_CSV, index=False)

print("Wrote", OUT_JSONL, "and", OUT_CSV)
out_df.head(10)

  0%|          | 0/52 [00:00<?, ?it/s]

Wrote llm_feature_tags.jsonl and llm_feature_tags.csv


,product_id,product_name,product_url,seamless,fabric_softness,wool_present,texture_level,pressure_level,adaptive_or_sensory_claim,seamless_evidence,fabric_softness_evidence,wool_present_evidence,texture_level_evidence,pressure_level_evidence,adaptive_claim_evidence,confidence,review_notes
0,1,Abby & Noah Sensory Compression Vest for Women...,https://sensoryfair.com/product/abby-noah-sens...,no,medium,no,unknown,medium,yes,Built to last with reinforced seams,"snug, comfortable fit",full material composition is not listed and wo...,,"calming, adjustable compression | customize th...","ideal for adults and children with ADHD, Anxie...",0.9,
1,2,Snugabye Adaptive Back Zip Convert-A-Foot Slee...,https://sensoryfair.com/product/adaptive-back-...,no,high,no,unknown,unknown,yes,The sleeper also features a back zipper,it is super soft and warm,Made from 100% polyester fleece,,,designed with your special needs child in mind,1.0,
2,3,"CBObaby Adaptive Bodysuit, Short Sleeve, Front...",https://sensoryfair.com/product/adaptive-bodys...,yes,high,no,unknown,medium,yes,tagless design and flat seams minimize sensory...,100% cotton construction offers supreme softness,100% cotton construction,,maintaining a secure fit throughout daily acti...,thoughtfully designed for children and teens w...,1.0,
3,4,EEOST Adaptive Clothing for Kids Special Needs...,https://sensoryfair.com/product/adaptive-cloth...,unknown,unknown,unknown,unknown,unknown,yes,,,,,,adaptive clothing | sensory-friendly,0.0,
4,5,Benefit Wear Anti-Strip Jumpsuit Clothing for ...,https://sensoryfair.com/product/benefit-wear-a...,unknown,medium,no,unknown,unknown,yes,,high-quality cotton/polyester blend knit fabri...,high-quality cotton/polyester blend,,,"perfect for kids with autism, tactile sensitiv...",0.9,
5,6,JOF Berlin Body Zipper For Kids | Blue | Onesi...,https://sensoryfair.com/product/berlin-body-zi...,unknown,unknown,unknown,unknown,unknown,yes,,,,,,adaptive clothing for children with special needs,0.0,
6,7,"Hugsmiling Body Sock Sensory Kids, Medium 27 *...",https://sensoryfair.com/product/body-sock-sens...,unknown,medium,no,unknown,medium,yes,,cozy and breathable,"Material: Matte Nylon Spandex Blend, Nylon 85%...",,deep pressure experience,great therapy and play tool for autism and sen...,1.0,
7,8,"Hugsmiling Body Sock Sensory Kids, Small 24 * ...",https://sensoryfair.com/product/body-sock-sens...,unknown,medium,no,unknown,medium,yes,,cozy and breathable,"Material: Matte Nylon Spandex Blend, Nylon 85%...",,deep pressure experience,great therapy and play tool for autism and sen...,0.9,
8,9,DC Comics Justice League 2 Pack Adaptive T-Shi...,https://sensoryfair.com/product/dc-comics-just...,yes,high,no,unknown,unknown,yes,flat seams | tag free design,soft 100% cotton | comfortable,100% cotton,,,designed to be comfortable for kids with senso...,1.0,
9,10,TescoRu Exclusive Back-Zip Onesie – Designed w...,https://sensoryfair.com/product/exclusive-back...,unknown,unknown,no,unknown,medium,unknown,,,Its stretchy fabric accommodates wear over clo...,,designed to offer a snug embrace | consider op...,,0.9,


## If you still get 429

Increase pacing:
- set `BASE_DELAY_S = 2.0` or `3.0`
- keep `N_LIMIT` small at first
- run later or request higher rate limits in your OpenAI account

429 is not a code bug; it means the API wants you to slow down.
